# Importing Libraries
**For more examples of what Kosh can do visit [GitHub Examples](https://github.com/LLNL/kosh/tree/stable/examples).**

In [ ]:
from numbers import Number
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns
import kosh
import math
import statistics
import numpy as np
from PIL import Image
import os
import sys

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

import graphviz

store = kosh.connect("my_store.sqlite", delete_all_contents=True)
print("Kosh is ready!")

# Loading Data

As mentioned in the `README.md`, the data we will be using is [Heart Failure Prediction Dataset (Classification, Label: HeartDisease)](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) and regression [Medical Cost Personal Datasets (Regression, Label: charges)](https://www.kaggle.com/datasets/mirichoi0218/insurance). We will load this data using Pandas DataFrames.


In [ ]:
##################
# Classification #
##################
df_classification = pd.read_csv("heart.csv")

##############
# Regression #
##############
df_regression = pd.read_csv("insurance.csv")

# Creating Kosh Ensemble
We create our Kosh Ensemble and add whatever metadata we want. The metadata (which end up being ensemble/dataset attributes) can later be used to find and filter specific ensembles/datasets. See [kosh/examples/Example_Simulation_Workflow.ipynb](https://github.com/LLNL/kosh/blob/stable/examples/Example_Simulation_Workflow.ipynb) for more information on how to add metadata, update it, and extract it.

A Kosh Ensemble is used to group multiple Kosh Datasets together. This is useful as we can later find all the data that is considered part of the same study. Here will create two Kosh Ensembles: one for the classification data and one for the regression data.

Note that `id`, `name` and `creator` are special named Kosh Ensemble/Dataset attributes. The attributes of an ensemble/dataset can be seen with `ensemble/dataset.list_attributes()` and extracted via `ensemble/dataset.MY_ATTRIBUTE`.

In [ ]:
##################
# Classification #
##################
metadata_classification = {"name": "Heart Failure Prediction Dataset",  # 'name' is a special named Kosh attribute for name of Kosh Dataset
                            "location": "https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction",
                           # add other metadata as needed
                           }

ensemble_classification = store.create_ensemble(metadata=metadata_classification)

##############
# Regression #
##############
metadata_regression = {"name": "Medical Cost Personal Datasets",  # 'name' is a special named Kosh attribute for name of Kosh Dataset
                            "location": "https://www.kaggle.com/datasets/mirichoi0218/insurance",
                           # add other metadata as needed
                           }

ensemble_regression = store.create_ensemble(metadata=metadata_regression)

# Creating Kosh Dataset

Here we create our Kosh Datasets: one for each row in the CSV file. This seems redundant since we already have a CSV file but imagine if this data took weeks, months, and/or years to gather so we would be creating these Kosh Datasets separately. Kosh makes it easy to save the data and update it as needed. As mentioned above, see [kosh/examples/Example_Simulation_Workflow.ipynb](https://github.com/LLNL/kosh/blob/stable/examples/Example_Simulation_Workflow.ipynb) for more information on Kosh.

In [ ]:
##################
# Classification #
##################
print("Classification Data:\n\n")
meta_list_classification = df_classification.to_dict('records')

for i, metadata in enumerate(meta_list_classification):
    dataset = ensemble_classification.create(metadata=metadata)
    if i == 0:
        print(metadata)
        print(dataset)
            
##############
# Regression #
##############
print("Regression Data:\n\n")
meta_list_regression = df_regression.to_dict('records')

for i, metadata in enumerate(meta_list_regression):
    dataset = ensemble_regression.create(metadata=metadata)
    if i == 0:
        print(metadata)
        print(dataset)

# Extracting Data

We can extract all the attributes for all Kosh Datasets in a Kosh Ensemble into a Pandas DataFrame using `ensemble.to_dataframe()`. By default this dataframe will always have the `id`, `name`, and `creator` columns. It also comes with the ensemble attributes formatted `{ensemble.id}_ENSEMBLE_ATTRIBUTE_{ensemble.attribute}`.


In [ ]:
##################
# Classification #
##################
df_classification_ensemble = ensemble_classification.to_dataframe()
print("\n\nPANDAS DATAFRAME Classifcation Ensemble:\n\n", df_classification_ensemble)

##############
# Regression #
##############
df_regression_ensemble = ensemble_regression.to_dataframe()
print("\n\nPANDAS DATAFRAME Regression Ensemble:\n\n", df_regression_ensemble)



# Splitting Data

We will extract our features and labels of interest from the dataframe above since `ensemble.to_dataframe()` also includes other metadata by default. We use SciKit Learn's `train_test_split()` method to split the data into train, validation, and test data.

No need to scale the data since decision trees use logical if statements instead of gradient descent.

In [ ]:
##################
# Classification #
##################
# {'Age': 40, 'Sex': 'M', 'ChestPainType': 'ATA', 'RestingBP': 140, 'Cholesterol': 289, 'FastingBS': 0, 'RestingECG': 'Normal', 'MaxHR': 172, 'ExerciseAngina': 'N', 'Oldpeak': 0.0, 'ST_Slope': 'Up', 'HeartDisease': 0}
classification_features = ['Age',
                           'Sex',
                           'ChestPainType',
                           'RestingBP', 'Cholesterol',
                           'FastingBS',
                           'RestingECG',
                           'MaxHR',
                           'ExerciseAngina',
                           'Oldpeak',
                           'ST_Slope']
classification_label = 'HeartDisease'

df_classification_ensemble_features = df_classification_ensemble[classification_features].copy()
df_classification_ensemble_labels = df_classification_ensemble[classification_label].copy()

# Splitting data
df_classification_ensemble_train_features, df_classification_ensemble_test_features, df_classification_ensemble_train_labels, df_classification_ensemble_test_labels = train_test_split(df_classification_ensemble_features, df_classification_ensemble_labels, test_size=0.2, random_state=42)
df_classification_ensemble_train_features, df_classification_ensemble_validation_features, df_classification_ensemble_train_labels, df_classification_ensemble_validation_labels = train_test_split(df_classification_ensemble_train_features, df_classification_ensemble_train_labels, test_size=0.2, random_state=42)

print("Classification Data:\n\n")
print(df_classification_ensemble_train_features.head())

print(f"Train Size features: {df_classification_ensemble_train_features.shape} and labels: {df_classification_ensemble_train_labels.shape}")
print(f"Validation Size features: {df_classification_ensemble_validation_features.shape} and labels: {df_classification_ensemble_validation_labels.shape}")
print(f"Test Size features: {df_classification_ensemble_test_features.shape} and labels: {df_classification_ensemble_test_labels.shape}")

##############
# Regression #
##############
# {'age': 19, 'sex': 'female', 'bmi': 27.9, 'children': 0, 'smoker': 'yes', 'region': 'southwest', 'charges': 16884.924}
regression_features = ['age',
                       'sex',
                       'bmi',
                       'children',
                       'smoker',
                       'region']
regression_label = 'charges'

df_regression_ensemble_features = df_regression_ensemble[regression_features].copy()
df_regression_ensemble_labels = df_regression_ensemble[regression_label].copy()

# Splitting data
df_regression_ensemble_train_features, df_regression_ensemble_test_features, df_regression_ensemble_train_labels, df_regression_ensemble_test_labels = train_test_split(df_regression_ensemble_features, df_regression_ensemble_labels, test_size=0.2, random_state=42)
df_regression_ensemble_train_features, df_regression_ensemble_validation_features, df_regression_ensemble_train_labels, df_regression_ensemble_validation_labels = train_test_split(df_regression_ensemble_train_features, df_regression_ensemble_train_labels, test_size=0.2, random_state=42)

print("\n\nRegression Data:\n\n")
print(df_regression_ensemble_train_features.head())

print(f"Train Size features: {df_regression_ensemble_train_features.shape} and labels: {df_regression_ensemble_train_labels.shape}")
print(f"Validation Size features: {df_regression_ensemble_validation_features.shape} and labels: {df_regression_ensemble_validation_labels.shape}")
print(f"Test Size features: {df_regression_ensemble_test_features.shape} and labels: {df_regression_ensemble_test_labels.shape}")

# Encode String Data

In order for a computer to "understand" string data, we need to encode it to numerical data. Below we see the unique values of the categorical columns.

In [ ]:
##################
# Classification #
##################
print("Classification Data:\n\n")
classification_cat_columns = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
for cat_column in classification_cat_columns:
    print(f"{cat_column}:", df_classification_ensemble_train_features[cat_column].unique())

##############
# Regression #
##############
print("\n\nRegression Data:\n\n")
regression_cat_columns = ['sex', 'smoker', 'region']
for cat_column in regression_cat_columns:
    print(f"{cat_column}:", df_regression_ensemble_train_features[cat_column].unique())
    


## Ordinal Data


For ordinal data where order does matter we use OrdinalEncoder. Each feature category will be assigned a numeric value and maintain its column name. Can also be used for Binary data since it is either a 0 or 1.

In [ ]:
##################
# Classification #
##################
classification_cat_columns_ordinal = ['Sex', 'RestingECG', 'ExerciseAngina']

category_orders_classification = [
    ['M', 'F'],  # 'Sex'
    ['Normal', 'ST', 'LVH'],   # 'RestingECG'
    ['N', 'Y']   # 'ExerciseAngina'
]

oe_classification = OrdinalEncoder(categories=category_orders_classification)

# Training set
df_classification_ensemble_train_features_oe = df_classification_ensemble_train_features.copy()
df_classification_ensemble_train_features_oe[classification_cat_columns_ordinal] = oe_classification.fit_transform(df_classification_ensemble_train_features_oe[classification_cat_columns_ordinal])

# Validation set
df_classification_ensemble_validation_features_oe = df_classification_ensemble_validation_features.copy()
df_classification_ensemble_validation_features_oe[classification_cat_columns_ordinal] = oe_classification.transform(df_classification_ensemble_validation_features_oe[classification_cat_columns_ordinal])

# Test set
df_classification_ensemble_test_features_oe = df_classification_ensemble_test_features.copy()
df_classification_ensemble_test_features_oe[classification_cat_columns_ordinal] = oe_classification.transform(df_classification_ensemble_test_features_oe[classification_cat_columns_ordinal])

print("Classification Data:\n\n")
print(df_classification_ensemble_train_features_oe)

##############
# Regression #
##############
regression_cat_columns_ordinal = ['sex', 'smoker',]

category_orders_regression = [
    ['male', 'female'],  # 'sex'
    ['no', 'yes']   # 'smoker'
]

oe_regression = OrdinalEncoder(categories=category_orders_regression)

# Training set
df_regression_ensemble_train_features_oe = df_regression_ensemble_train_features.copy()
df_regression_ensemble_train_features_oe[regression_cat_columns_ordinal] = oe_regression.fit_transform(df_regression_ensemble_train_features_oe[regression_cat_columns_ordinal])

# Validation set
df_regression_ensemble_validation_features_oe = df_regression_ensemble_validation_features.copy()
df_regression_ensemble_validation_features_oe[regression_cat_columns_ordinal] = oe_regression.transform(df_regression_ensemble_validation_features_oe[regression_cat_columns_ordinal])

# Test set
df_regression_ensemble_test_features_oe = df_regression_ensemble_test_features.copy()
df_regression_ensemble_test_features_oe[regression_cat_columns_ordinal] = oe_regression.transform(df_regression_ensemble_test_features_oe[regression_cat_columns_ordinal])

print("\n\nRegression Data:\n\n")
print(df_regression_ensemble_train_features_oe)

## Nominal Data

For nominal data where order doesn't matter we use OneHotEncoder. Each feature category will be assigned a new column with a binary value causing new features to be created.

In [ ]:
##################
# Classification #
##################
classification_cat_columns_nominal = ['ChestPainType', 'ST_Slope']

ohe_classification = OneHotEncoder(handle_unknown='ignore')

# Training set
df_classification_ensemble_train_features_oe_ohe = df_classification_ensemble_train_features_oe.copy()

ohe_train = ohe_classification.fit_transform(df_classification_ensemble_train_features_oe_ohe[classification_cat_columns_nominal]).toarray()
ohe_feature_names = ohe_classification.get_feature_names_out(classification_cat_columns_nominal)

df_train_ohe = pd.DataFrame(ohe_train, columns=ohe_feature_names, index=df_classification_ensemble_train_features_oe_ohe.index)

df_classification_ensemble_train_features_oe_ohe = pd.concat([
    df_classification_ensemble_train_features_oe_ohe.drop(classification_cat_columns_nominal, axis=1),
    df_train_ohe
], axis=1)


# Validation set
df_classification_ensemble_validation_features_oe_ohe = df_classification_ensemble_validation_features_oe.copy()

ohe_validation = ohe_classification.transform(df_classification_ensemble_validation_features_oe_ohe[classification_cat_columns_nominal]).toarray()

df_validation_ohe = pd.DataFrame(ohe_validation, columns=ohe_feature_names, index=df_classification_ensemble_validation_features_oe_ohe.index)

df_classification_ensemble_validation_features_oe_ohe = pd.concat([
    df_classification_ensemble_validation_features_oe_ohe.drop(classification_cat_columns_nominal, axis=1),
    df_validation_ohe
], axis=1)

# Test set
df_classification_ensemble_test_features_oe_ohe = df_classification_ensemble_test_features_oe.copy()

ohe_test = ohe_classification.transform(df_classification_ensemble_test_features_oe_ohe[classification_cat_columns_nominal]).toarray()

df_test_ohe = pd.DataFrame(ohe_test, columns=ohe_feature_names, index=df_classification_ensemble_test_features_oe_ohe.index)

df_classification_ensemble_test_features_oe_ohe = pd.concat([
    df_classification_ensemble_test_features_oe_ohe.drop(classification_cat_columns_nominal, axis=1),
    df_test_ohe
], axis=1)

print("Classification Data:\n\n")
print(df_classification_ensemble_train_features_oe_ohe)


##############
# Regression #
##############
regression_cat_columns_nominal = ['region']

ohe_regression = OneHotEncoder(handle_unknown='ignore')

# Training set
df_regression_ensemble_train_features_oe_ohe = df_regression_ensemble_train_features_oe.copy()

ohe_train = ohe_regression.fit_transform(df_regression_ensemble_train_features_oe_ohe[regression_cat_columns_nominal]).toarray()
ohe_feature_names = ohe_regression.get_feature_names_out(regression_cat_columns_nominal)

df_train_ohe = pd.DataFrame(ohe_train, columns=ohe_feature_names, index=df_regression_ensemble_train_features_oe_ohe.index)

df_regression_ensemble_train_features_oe_ohe = pd.concat([
    df_regression_ensemble_train_features_oe_ohe.drop(regression_cat_columns_nominal, axis=1),
    df_train_ohe
], axis=1)


# Validation set
df_regression_ensemble_validation_features_oe_ohe = df_regression_ensemble_validation_features_oe.copy()

ohe_validation = ohe_regression.transform(df_regression_ensemble_validation_features_oe_ohe[regression_cat_columns_nominal]).toarray()

df_validation_ohe = pd.DataFrame(ohe_validation, columns=ohe_feature_names, index=df_regression_ensemble_validation_features_oe_ohe.index)

df_regression_ensemble_validation_features_oe_ohe = pd.concat([
    df_regression_ensemble_validation_features_oe_ohe.drop(regression_cat_columns_nominal, axis=1),
    df_validation_ohe
], axis=1)

# Test set
df_regression_ensemble_test_features_oe_ohe = df_regression_ensemble_test_features_oe.copy()

ohe_test = ohe_regression.transform(df_regression_ensemble_test_features_oe_ohe[regression_cat_columns_nominal]).toarray()

df_test_ohe = pd.DataFrame(ohe_test, columns=ohe_feature_names, index=df_regression_ensemble_test_features_oe_ohe.index)

df_regression_ensemble_test_features_oe_ohe = pd.concat([
    df_regression_ensemble_test_features_oe_ohe.drop(regression_cat_columns_nominal, axis=1),
    df_test_ohe
], axis=1)

print("Regression Data:\n\n")
print(df_regression_ensemble_train_features_oe_ohe)

# Turning Pandas DataFrames into Matricies

Now we will convert our Pandas DataFrames into matricies so the Machine Learning algorithms can process the data.

In [ ]:
##################
# Classification #
##################
X_train_classification = df_classification_ensemble_train_features_oe_ohe.to_numpy()
y_train_classification = df_classification_ensemble_train_labels.to_numpy()

X_validation_classification = df_classification_ensemble_validation_features_oe_ohe.to_numpy()
y_validation_classification = df_classification_ensemble_validation_labels.to_numpy()

X_test_classification = df_classification_ensemble_test_features_oe_ohe.to_numpy()
y_test_classification = df_classification_ensemble_test_labels.to_numpy()

print("Classification Data:\n\n")
print(X_train_classification)
print(y_train_classification)

##############
# Regression #
##############
X_train_regression = df_regression_ensemble_train_features_oe_ohe.to_numpy()
y_train_regression = df_regression_ensemble_train_labels.to_numpy()

X_validation_regression = df_regression_ensemble_validation_features_oe_ohe.to_numpy()
y_validation_regression = df_regression_ensemble_validation_labels.to_numpy()

X_test_regression = df_regression_ensemble_test_features_oe_ohe.to_numpy()
y_test_regression = df_regression_ensemble_test_labels.to_numpy()

print("\n\nRegression Data:\n\n")
print(X_train_regression)
print(y_train_regression)

# Train The Model

We will now train our model using `sklearn.tree.DecisionTreeClassifier()` and `sklearntree.DecisionTreeRegressor()`. We will use `graphviz` to export the full tree as a png/pdf.

Notice how the regression tree has a lot more nodes and branches. This is because it is continous data and not categorical data so it has to create a lot more logical if statements.

In [ ]:
##################
# Classification #
##################
classification_tree = tree.DecisionTreeClassifier()
classification_tree.fit(X_train_classification, y_train_classification)

fig, ax = plt.subplots(figsize=(10,10))
tree.plot_tree(classification_tree, max_depth=1,
               feature_names=df_classification_ensemble_train_features_oe_ohe.columns,
               class_names = ['HeartDisease_No', 'HeartDisease_Yes'],
               filled=True,
               proportion=True,
               ax=ax
              )

classification_tree_dot = tree.export_graphviz(
    classification_tree,
    feature_names=df_classification_ensemble_train_features_oe_ohe.columns,
    class_names=['HeartDisease_No', 'HeartDisease_Yes'],
    filled=True,
    proportion=True,
)

graph = graphviz.Source(classification_tree_dot)
graph.format = 'png'
graph.render('classification_tree')

depth = classification_tree.get_depth()
leaves = classification_tree.get_n_leaves()
nodes = classification_tree.tree_.node_count
branches = nodes - 1

print("Classification Data:\n\n")
print(f"\tDepth: {depth}")
print(f"\tLeaves: {leaves}")
print(f"\tNodes: {nodes}")
print(f"\tBranches: {branches}")

##############
# Regression #
##############
regression_tree = tree.DecisionTreeRegressor()
regression_tree.fit(X_train_regression, y_train_regression)

fig, ax = plt.subplots(figsize=(10,10))
tree.plot_tree(regression_tree, max_depth=1,
               feature_names=df_regression_ensemble_train_features_oe_ohe.columns,
               class_names = ['charges'],
               filled=True,
               proportion=True,
               ax=ax
              )

regression_tree_dot = tree.export_graphviz(
    regression_tree,
               feature_names=df_regression_ensemble_train_features_oe_ohe.columns,
               class_names = ['charges'],
               filled=True,
               proportion=True,
              )

graph = graphviz.Source(regression_tree_dot)
graph.format = 'png'
graph.render('regression_tree')

depth = regression_tree.get_depth()
leaves = regression_tree.get_n_leaves()
nodes = regression_tree.tree_.node_count
branches = nodes - 1

print("\n\nRegression Data:\n\n")
print(f"\tDepth: {depth}")
print(f"\tLeaves: {leaves}")
print(f"\tNodes: {nodes}")
print(f"\tBranches: {branches}")

# Inference and Confusion Matrix

Now that our model is trained, we can calculate the score which is just the mean accuracy `sklearn.tree.DecisionTreeClassifier.score()` for classification and coefficient of determination `sklearntree.DecisionTreeRegressor.score()` for regression for our train, validation, and test data. We can also see what the model will infer/predict using `sklearn.linear_model.<tree>.predict()` and plot them via the confusion matrix `sklearn.metrics.confusion_matrix()` for classification or scatter plot for regression.

The Train Data is pretty much perfect. However, this is misleading since once you see the score for the Validation Data and Test Data, the algorithm is clearly overfitting. One of the reasons for overfitting is we are using the default `max_depth=None` which means the tree keeps on splitting until there is one data point per end node or leaf. This is visually evident in the image of the full tree saved as a png. Setting a `max_depth` will help with ovefitting since the logical if statements won't become so granular and the prediction will just be the most frequent class for classification or the average value for regression of that leaf.

In [ ]:
def scores_and_inference(model,
                         X_train, X_validation, X_test,
                         y_train, y_validation, y_test):
    if isinstance(model, tree.DecisionTreeClassifier):
        score = "Mean Accuracy"
    elif isinstance(model, tree.DecisionTreeRegressor):
        score = "Coefficient of Determination"

    for X, y, data_type in zip([X_train, X_validation, X_test],
                               [y_train, y_validation, y_test],
                               ['Train_Data', 'Validation_Data', 'Test_Data']):
        print(f"{data_type} Mean Accuracy:", model.score(X, y))

        print(f'Train Prediction {model.predict(X[-1].reshape(1,-1))[0]} and actual value {y[-1]}')
        predictions = model.predict(X)

        if isinstance(model, tree.DecisionTreeClassifier):
            cm = confusion_matrix(y, predictions, labels=model.classes_)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                          display_labels=model.classes_)
            disp.plot()
            plt.title(data_type)
            plt.savefig(f'classification_{data_type}_confusion_matrix.png')
            plt.show()
        elif isinstance(model, tree.DecisionTreeRegressor):
            plt.scatter(y, predictions)
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(data_type)
            plt.savefig(f'regression_{data_type}_scatter_plot.png')
            plt.show()


print("Classification Data:\n\n")
scores_and_inference(classification_tree,
                         X_train_classification, X_validation_classification, X_test_classification,
                         y_train_classification, y_validation_classification, y_test_classification)

print("\n\nRegression Data:\n\n")

scores_and_inference(regression_tree,
                         X_train_regression, X_validation_regression, X_test_regression,
                         y_train_regression, y_validation_regression, y_test_regression)